# Distance Detector — Entraînement YOLOv8
**Auteur** : MBIDA NGUELE Paul Loïc  
**Projet** : Vision Assistant — ENSPY ANI-IA  

Ce notebook entraîne un modèle YOLOv8 pour estimer si une personne est **proche** ou **loin** de la caméra.




In [1]:
# Vérifier le GPU
!nvidia-smi

Mon Jun 29 06:50:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Installer les dépendances
!pip install ultralytics roboflow --quiet
print(' Dépendances installées')

 Dépendances installées


In [5]:
# Télécharger le dataset Roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="X0GHwX6lC6ELRgrqvOc8")
project = rf.workspace("realize").project("person-detection-svczk")
version = project.version(1)
dataset = version.download("yolov8")


DATASET_PATH = dataset.location
print(f' Dataset téléchargé : {DATASET_PATH}')

loading Roboflow workspace...
loading Roboflow project...
 Dataset téléchargé : /content/Person-detection-1


In [7]:
# Re-labelliser : person → close_person / far_person
#
# Critère : si la hauteur de la bbox > 30% de l'image → close
#           sinon → far
import os
import glob

CLOSE_THRESHOLD = 0.30  # hauteur bbox relative > 30% = proche

def relabel_split(labels_dir):
    """Re-labellise tous les fichiers .txt d'un split (train ou val)."""
    txt_files = glob.glob(os.path.join(labels_dir, '*.txt'))
    close_count = 0
    far_count = 0

    for txt_path in txt_files:
        new_lines = []
        with open(txt_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 5:
                    continue

                # Format YOLO : class x_center y_center width height
                # On ignore l'ancienne classe (tous sont 'person' = 0)
                _, x_c, y_c, w, h = parts[:5]
                h = float(h)

                # Nouvelle classe selon la taille
                if h > CLOSE_THRESHOLD:
                    new_class = 0  # close_person
                    close_count += 1
                else:
                    new_class = 1  # far_person
                    far_count += 1

                new_lines.append(f"{new_class} {x_c} {y_c} {w} {h}")

        with open(txt_path, 'w') as f:
            f.write('\n'.join(new_lines))

    return close_count, far_count

# Appliquer sur train et val
train_labels = os.path.join(DATASET_PATH, 'train', 'labels')
val_labels   = os.path.join(DATASET_PATH, 'valid', 'labels')

c1, f1 = relabel_split(train_labels)
c2, f2 = relabel_split(val_labels)

print(f' Re-labellisation terminée')
print(f'   Train → close_person: {c1} | far_person: {f1}')
print(f'   Val   → close_person: {c2} | far_person: {f2}')

 Re-labellisation terminée
   Train → close_person: 27 | far_person: 237
   Val   → close_person: 5 | far_person: 6


In [8]:
# Créer le data.yaml adapté au dataset Roboflow
import yaml

data_config = {
    'path': DATASET_PATH,
    'train': 'train/images',
    'val':   'valid/images',
    'nc': 2,
    'names': ['close_person', 'far_person']
}

yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f' data.yaml créé : {yaml_path}')
print(open(yaml_path).read())

 data.yaml créé : /content/Person-detection-1/data.yaml
names:
- close_person
- far_person
nc: 2
path: /content/Person-detection-1
train: train/images
val: valid/images



In [9]:
# Entraînement YOLOv8
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano — léger, rapide, idéal mobile

results = model.train(
    data      = yaml_path,
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    patience  = 10,
    project   = '/content/runs',
    name      = 'distance_detector',
    exist_ok  = True,
    pretrained= True,
    optimizer = 'Adam',
    lr0       = 0.001,
    augment   = True,
    verbose   = True,
)

print('\n Entraînement terminé !')

Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Person-detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=distance_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_

In [10]:
# Vérifier les résultats
best_pt = '/content/runs/distance_detector/weights/best.pt'

import os
if os.path.exists(best_pt):
    size_mb = os.path.getsize(best_pt) / 1024 / 1024
    print(f'✅ best.pt trouvé : {size_mb:.1f} MB')
else:
    print('❌ best.pt introuvable — vérifiez l\'entraînement')

✅ best.pt trouvé : 6.0 MB


In [11]:
# Export ONNX
from ultralytics import YOLO

model = YOLO(best_pt)
model.export(
    format   = 'onnx',
    imgsz    = 640,
    dynamic  = False,
    simplify = True,
    opset    = 12,
)

best_onnx = best_pt.replace('.pt', '.onnx')
print(f' Export ONNX : {best_onnx}')

Ultralytics 8.4.82 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/distance_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 416ms
Prepared 4 packages in 1.85s
Installed 4 packages in 244ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming 

In [12]:
# Télécharger best.pt et best.onnx
from google.colab import files

print('Téléchargement de best.pt ...')
files.download(best_pt)

print('Téléchargement de best.onnx ...')
files.download(best_onnx)

print('✅ Téléchargements lancés — vérifie ton dossier Téléchargements')

Téléchargement de best.pt ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Téléchargement de best.onnx ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargements lancés — vérifie ton dossier Téléchargements
